# 00 Data Preprocessing

## Imports

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.plotting import heatmap

import tulipy as ti

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras import initializers
import keras_tuner as kt
from keras_tuner import HyperParameters

import warnings

## Data Retrieval

In [2]:
# Set Scope of Date
start_date_analyzed = '2021-01-01'
end_date_analyzed = '2025-01-01'

# Retrieve S&P 500 tickers
sp500_tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
tickers = sp500_tables[0].Symbol  # Storing just the symbols into a list
tickers_changed = sp500_tables[1]  # Including the changed tickers as well to remove survivorship bias

# Getting only the added and removed tickers
tickers_added = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Added
tickers_removed = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Removed

# Removing the additional added tickers
tickers = tickers[~(tickers.isin(tickers_added.Ticker))]

# Appending the removed tickers
tickers = pd.concat([tickers, tickers_removed.Ticker], ignore_index=True)

# Dropping any NA values
tickers = tickers.dropna()

# Dropping duplicates
tickers = tickers.drop_duplicates()

# Resetting the index
tickers = tickers.reset_index(drop = True)

# Filtering out problematic tickers
problematic_tickers = ['BF.B', 'BRK.B', 'GEV', 'SOLV', 'SW', 'VLTO']
filtered_tickers = [ticker for ticker in tickers if ticker not in problematic_tickers]

len(filtered_tickers)

508

In [3]:
# One note: don't scrap too many times in a row from Yahoo Finance, or they might block you

In [4]:
# Getting Data for S&P500 Stocks
multi_data = yf.download(filtered_tickers, start=start_date_analyzed, end=end_date_analyzed)

# NOTE: there will be missing stocks because Yahoo Finance has delisted some of the stocks that went out of the S&P500 (like SIVB)
multi_data

[*********************100%***********************]  508 of 508 completed

26 Failed downloads:
['ATVI', 'NLSN', 'SIVB', 'ALXN', 'ABMD', 'FRC', 'DRE', 'CXO', 'KSU', 'MXIM', 'TWTR', 'CERN', 'MRO', 'FLIR', 'TIF', 'FBHS', 'PXD', 'PBCT', 'DISH', 'XLNX', 'HFC', 'VAR', 'GPS', 'CTXS']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
['ETR', 'LEN']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01)')


Price      Adj Close                                            ... Volume  \
Ticker          ABMD ALXN ATVI CERN CTXS CXO DISH DRE ETR FBHS  ...   XLNX   
Date                                                            ...          
2021-01-04       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2021-01-05       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2021-01-06       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2021-01-07       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2021-01-08       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
...              ...  ...  ...  ...  ...  ..  ...  ..  ..  ...  ...    ...   
2024-12-24       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2024-12-26       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2024-12-27       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2024-12-30       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   
2024-12-31       NaN  NaN  NaN  NaN  NaN NaN  NaN NaN NaN  NaN  ...    NaN   

Price                                                                      \
Ticker           XOM     XRAY      XRX      XYL      YUM      ZBH    ZBRA   
Date                                                                        
2021-01-04  27764700  2214500  1764900   814700  2058200  1265870  432600   
2021-01-05  44035100  3021700  1297800  1005200  1475300   998894  374700   
2021-01-06  36484800  1774700  2276900  1015300  1352700  1424799  419500   
2021-01-07  29528100  1334200  1737500  1539300  2222900  1237648  379900   
2021-01-08  28628200  1336100  1554800  1632200  1252200  1259072  348500   
...              ...      ...      ...      ...      ...      ...     ...   
2024-12-24   7807000   964900  2157000   379300   533000   458600   88700   
2024-12-26   9652400  2577100  2971100   575700  1040900  1277300  140100   
2024-12-27  11943900  1876900  1874700   552400  1146300   743400  287200   
2024-12-30  11080800  1965500  2120300   586800  1144600  1532000  211300   
2024-12-31  12387800  1595200  2035800   641600  1217100   683300  327900   

Price                         
Ticker         ZION      ZTS  
Date                          
2021-01-04  1625200  1805900  
2021-01-05  1349500  1358600  
2021-01-06  3269500  2139800  
2021-01-07  2333900  1710700  
2021-01-08  2201000  1475900  
...             ...      ...  
2024-12-24   371100  1023600  
2024-12-26  1158200  2167200  
2024-12-27   923400  1800100  
2024-12-30   827800  1531400  
2024-12-31  1426000  1327400  

[1005 rows x 2566 columns]

## Data Cleaning

In [5]:
# Dropping nulls based on column:
cleaned_multi_data = multi_data.dropna(axis=1)

cleaned_multi_data

Price            Close                                                 \
Ticker               A        AAL         AAP        AAPL        ABBV   
Date                                                                    
2021-01-04  115.582756  15.130000  143.031052  126.405251   88.933968   
2021-01-05  116.527748  15.430000  142.876526  127.968094   89.853607   
2021-01-06  119.723221  15.520000  151.130737  123.660477   89.077385   
2021-01-07  122.908966  15.380000  152.421600  127.880173   90.030762   
2021-01-08  123.785782  15.130000  154.594238  128.983978   90.503227   
...                ...        ...         ...         ...         ...   
2024-12-24  135.848907  17.350000   43.489883  257.916443  178.318420   
2024-12-26  135.579407  17.350000   43.997086  258.735504  177.525894   
2024-12-27  135.289932  17.350000   43.818073  255.309296  176.347015   
2024-12-30  134.171997  17.620001   45.797142  251.923019  174.553925   
2024-12-31  134.339996  17.430000   47.030334  250.144974  176.039917   

Price                                                                  ...  \
Ticker             ABT         ACN        ADBE         ADI        ADM  ...   
Date                                                                   ...   
2021-01-04  101.045967  242.118484  485.339996  136.015427  45.076885  ...   
2021-01-05  102.296181  243.496872  485.690002  138.446609  45.953655  ...   
2021-01-06  102.083199  246.159149  466.309998  139.070694  47.815678  ...   
2021-01-07  103.074104  248.481628  477.739990  144.948395  47.390835  ...   
2021-01-08  103.361191  249.387848  485.100006  146.000931  47.517391  ...   
...                ...         ...         ...         ...        ...  ...   
2024-12-24  114.160919  361.630005  447.940002  218.089996  50.630001  ...   
2024-12-26  114.668251  360.429993  450.160004  218.059998  50.610001  ...   
2024-12-27  114.389717  356.179993  446.480011  216.990005  50.580002  ...   
2024-12-30  112.211151  352.489990  445.799988  212.559998  50.070000  ...   
2024-12-31  112.519531  351.790009  444.679993  212.460007  50.520000  ...   

Price        Volume                                                         \
Ticker          XEL       XOM     XRAY      XRX      XYL      YUM      ZBH   
Date                                                                         
2021-01-04  3197100  27764700  2214500  1764900   814700  2058200  1265870   
2021-01-05  1976800  44035100  3021700  1297800  1005200  1475300   998894   
2021-01-06  2667000  36484800  1774700  2276900  1015300  1352700  1424799   
2021-01-07  2630400  29528100  1334200  1737500  1539300  2222900  1237648   
2021-01-08  2221500  28628200  1336100  1554800  1632200  1252200  1259072   
...             ...       ...      ...      ...      ...      ...      ...   
2024-12-24   943900   7807000   964900  2157000   379300   533000   458600   
2024-12-26  1394900   9652400  2577100  2971100   575700  1040900  1277300   
2024-12-27  2015000  11943900  1876900  1874700   552400  1146300   743400   
2024-12-30  2642900  11080800  1965500  2120300   586800  1144600  1532000   
2024-12-31  2143800  12387800  1595200  2035800   641600  1217100   683300   

Price                                 
Ticker        ZBRA     ZION      ZTS  
Date                                  
2021-01-04  432600  1625200  1805900  
2021-01-05  374700  1349500  1358600  
2021-01-06  419500  3269500  2139800  
2021-01-07  379900  2333900  1710700  
2021-01-08  348500  2201000  1475900  
...            ...      ...      ...  
2024-12-24   88700   371100  1023600  
2024-12-26  140100  1158200  2167200  
2024-12-27  287200   923400  1800100  
2024-12-30  211300   827800  1531400  
2024-12-31  327900  1426000  1327400  

[1005 rows x 2380 columns]

In [6]:
# Making cleaned_multi_data so that there are rows for each ticker for each date:
unstacked = cleaned_multi_data.stack(level=0).unstack(level=0)
unstacked = unstacked.transpose().reset_index()
unstacked.columns.name = None

unstacked

,Ticker,Date,Close,High,Low,Open,Volume
0,A,2021-01-04,115.582756,116.995388,114.092184,115.875028,2030700.0
1,A,2021-01-05,116.527748,117.219451,115.027432,115.426870,2344000.0
2,A,2021-01-06,119.723221,120.687707,115.163820,115.631454,2428500.0
3,A,2021-01-07,122.908966,124.194950,120.911790,121.126122,1775900.0
4,A,2021-01-08,123.785782,124.350838,122.606963,123.230471,1577200.0
...,...,...,...,...,...,...,...
478375,ZTS,2024-12-24,164.203583,164.542554,162.249499,163.206597,1023600.0
478376,ZTS,2024-12-26,165.021118,165.290293,162.528653,163.226540,2167200.0
478377,ZTS,2024-12-27,164.103897,166.028073,163.027150,164.452833,1800100.0
478378,ZTS,2024-12-30,161.751007,163.555549,160.963380,163.396028,1531400.0


In [7]:
# Getting Date to datetime format and setting as index
unstacked['Date'] = pd.to_datetime(unstacked['Date'])
data=unstacked.sort_values(by="Ticker",ascending=True)
data.set_index("Date",inplace=True)

# Re-ordering as OHLCV (industry standard) format
data = data[['Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]

data.head()

,Ticker,Open,High,Low,Close,Volume
Date,,,,,,
2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0
2023-08-22,A,116.605188,117.357163,116.090684,116.397408,1422300.0
2023-08-23,A,117.258235,117.891474,116.466682,117.871681,1289300.0
2023-08-24,A,118.099243,119.504242,117.327483,118.227867,2045900.0
2023-08-25,A,118.534606,119.543829,117.268120,118.415871,1273300.0


In [8]:
# Creating a new Date_col column to rank and drop dates appropriately
data = data.reset_index().set_index("Date", drop=False)
data = data.rename(columns={"Date": "Date_col"})
data["rank"] = data.groupby("Ticker")["Date_col"].rank()
data = data.sort_values(by=["Ticker", "Date_col"])

data

,Date_col,Ticker,Open,High,Low,Close,Volume,rank
Date,,,,,,,,
2021-01-04,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0
2021-01-05,2021-01-05,A,115.426870,117.219451,115.027432,116.527748,2344000.0,2.0
2021-01-06,2021-01-06,A,115.631454,120.687707,115.163820,119.723221,2428500.0,3.0
2021-01-07,2021-01-07,A,121.126122,124.194950,120.911790,122.908966,1775900.0,4.0
2021-01-08,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0
...,...,...,...,...,...,...,...,...
2024-12-24,2024-12-24,ZTS,163.206597,164.542554,162.249499,164.203583,1023600.0,1001.0
2024-12-26,2024-12-26,ZTS,163.226540,165.290293,162.528653,165.021118,2167200.0,1002.0
2024-12-27,2024-12-27,ZTS,164.452833,166.028073,163.027150,164.103897,1800100.0,1003.0


# Getting cleaned S&P500 data into a csv file

In [385]:
# Dropping rank because not necessary
data_to_csv = data.iloc[:, :-1]

# Getting the data into a csv file
# data.to_csv('SP500_4y_day_data.csv')